### 1. Install Crop Simulation Utilities

To run an offline crop model, we can install libraries for handling agricultural data and predicting growth stages. We will use a Python-native implementation to simulate crop phenology dynamically.

In [ ]:
# Install libraries for handling spatial, weather, or crop model inputs
!pip install -q pandas numpy matplotlib

### 2. Conceptual Overview of Crop Calendar Inputs

The crop model simulates the physiological development of the crop on a daily basis using three key inputs:
1. **Weather Data:** Daily values of temperatures and rainfall.
2. **Soil Data (Specified Once):** Water storage capacity parameters.
3. **Crop Type:** Genetic coefficients defining how fast the crop matures based on temperature and day length.

In [1]:
import pandas as pd
import numpy as np

# Configure our Crop and Soil Parameters once
crop_parameters = {
    'Maize': {
        'base_temp': 10.0,
        'flowering_gdd': 650,
        'maturity_gdd': 1200
    },
    'Wheat': {
        'base_temp': 0.0,
        'flowering_gdd': 800,
        'maturity_gdd': 1400
    }
}

# Soil properties specified once (e.g., maximum water holding capacity in mm)
soil_parameters = {
    'max_water_capacity': 150.0, # mm of water the soil profile can hold
    'initial_water_content': 75.0 # mm of starting water
}

print("Crop parameters and soil properties configured successfully!")

Crop parameters and soil properties configured successfully!


### 3. Dynamic Crop Growth and Soil Water Balance Simulation

We will simulate a continuous daily engine that tracks both crop maturity (using heat accumulation) and soil moisture to output a detailed crop calendar.

In [2]:
def run_crop_simulation(weather_df, crop_type, soil_params):
    crop = crop_parameters[crop_type]
    t_base = crop['base_temp']

    # Tracking states
    cumulative_gdd = 0.0
    soil_water = soil_params['initial_water_content']
    max_water = soil_params['max_water_capacity']

    calendar = []
    flowering_date = None
    maturity_date = None

    for idx, row in weather_df.iterrows():
        # 1. Update Soil Water Balance (Rainfall in, Evapotranspiration/loss out)
        # For simplicity, daily loss is estimated at 3.5mm on warm days
        et_loss = 3.5 if (row['Tmax'] + row['Tmin'])/2 > 15 else 1.5
        soil_water = min(max_water, max(10.0, soil_water + row['Rainfall'] - et_loss))

        # 2. Update Crop Phenology (Heat Units)
        avg_temp = (row['Tmax'] + row['Tmin']) / 2
        daily_gdd = max(0.0, avg_temp - t_base)
        cumulative_gdd += daily_gdd

        # Check Phenology milestones
        stage = "Vegetative"
        if cumulative_gdd >= crop['maturity_gdd']:
            stage = "Maturity"
            if maturity_date is None:
                maturity_date = row['Date']
        elif cumulative_gdd >= crop['flowering_gdd']:
            stage = "Flowering"
            if flowering_date is None:
                flowering_date = row['Date']

        calendar.append({
            'Date': row['Date'],
            'Cumulative_GDD': cumulative_gdd,
            'Soil_Water_mm': soil_water,
            'Stage': stage,
            'Avg_Temp': avg_temp,
            'Rainfall': row['Rainfall'],
            'Humidity': row['Humidity']
        })

    return pd.DataFrame(calendar), flowering_date, maturity_date

print("Crop simulator function defined.")

Crop simulator function defined.


### 4. Pesticide and Fertilizer Application Rules

Using the simulator outputs, we can query optimal management dates using weather safety limits (e.g. low wind, no rain, proper crop stages).

In [3]:
def get_management_schedule(sim_df):
    # Fertilize at early vegetative stage (GDD between 200 and 400)
    # Apply on a day with light rainfall scheduled to help fertilizer integrate into soil
    fertilizer_days = sim_df[
        (sim_df['Cumulative_GDD'] >= 200) & (sim_df['Cumulative_GDD'] <= 400) &
        (sim_df['Rainfall'] > 1.0) & (sim_df['Rainfall'] < 10.0)
    ]

    # Spray Pesticides during flowering window (GDD 650 to 900) but only on dry days to avoid wash-off
    pesticide_days = sim_df[
        (sim_df['Cumulative_GDD'] >= 650) & (sim_df['Cumulative_GDD'] <= 900) &
        (sim_df['Rainfall'] == 0.0) & (sim_df['Humidity'] > 75.0)
    ]

    return {
        'fertilizer_dates': fertilizer_days['Date'].dt.strftime('%Y-%m-%d').tolist(),
        'pesticide_dates': pesticide_days['Date'].dt.strftime('%Y-%m-%d').tolist()
    }

print("Management scheduling rules loaded.")

Management scheduling rules loaded.


### 5. Fetching Real Weather Data via Open-Meteo API

Instead of simulating synthetic weather data, we can call the Open-Meteo API. Let's write a cell to pull actual historical daily weather (temperature, relative humidity, and rainfall) for a specific farming location (e.g., latitude 42.03, longitude -93.62) and run our crop model dynamically on real data.

In [ ]:
import requests
import pandas as pd

# Coordinates for Ames, Iowa (a major agricultural research location)
latitude = 42.03
longitude = -93.62
start_date = "2023-05-01"
end_date = "2023-08-31"

# API URL to fetch daily max/min temperatures, precipitation, and average relative humidity
url = f"https://archive-api.open-meteo.com/v1/archive?latitude={latitude}&longitude={longitude}&start_date={start_date}&end_date={end_date}&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_mean&timezone=auto"

response = requests.get(url)
if response.status_code == 200:
    data = response.json()
    daily_data = data['daily']

    # Load into a pandas DataFrame
    real_weather_df = pd.DataFrame({
        'Date': pd.to_datetime(daily_data['time']),
        'Tmax': daily_data['temperature_2m_max'],
        'Tmin': daily_data['temperature_2m_min'],
        'Rainfall': daily_data['precipitation_sum'],
        'Humidity': daily_data['relative_humidity_2m_mean']
    })

    print(f"Successfully fetched {len(real_weather_df)} days of real weather data!")
    display(real_weather_df.head())
else:
    print(f"Failed to fetch data. Status code: {response.status_code}")

Successfully fetched 123 days of real weather data!


,Date,Tmax,Tmin,Rainfall,Humidity
0,2023-05-01,14.9,5.3,0.0,37
1,2023-05-02,15.8,4.6,0.0,34
2,2023-05-03,19.5,2.4,0.0,39
3,2023-05-04,25.9,7.2,0.0,35
4,2023-05-05,22.2,10.4,8.1,71


### 5. Running the Simulation with Real Weather & Soil Data

In [7]:
# Let's simulate 'Maize' using the weather data pulled in the previous step
crop_selected = 'Maize'

try:
    # Ensure weather data is loaded locally
    df_to_use = real_weather_df
except NameError:
    import requests
    # Fallback to fetch Ames, Iowa 2023 weather data directly if the previous cell was skipped
    url = "https://archive-api.open-meteo.com/v1/archive?latitude=42.03&longitude=-93.62&start_date=2023-05-01&end_date=2023-08-31&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_mean&timezone=auto"
    response = requests.get(url).json()['daily']
    df_to_use = pd.DataFrame({
        'Date': pd.to_datetime(response['time']),
        'Tmax': response['temperature_2m_max'],
        'Tmin': response['temperature_2m_min'],
        'Rainfall': response['precipitation_sum'],
        'Humidity': response['relative_humidity_2m_mean']
    })
    real_weather_df = df_to_use

sim_results_df, flowering, maturity = run_crop_simulation(df_to_use, crop_selected, soil_parameters)
schedules = get_management_schedule(sim_results_df)

print(f"=== Run Summary for {crop_selected} ===")
if flowering:
    print(f"Predicted Flowering Date: {flowering.strftime('%Y-%m-%d')}")
if maturity:
    print(f"Predicted Maturity Date: {maturity.strftime('%Y-%m-%d')}")
print(f"Recommended Fertilizer Windows: {schedules['fertilizer_dates'][:3]}")
print(f"Recommended Pesticide Windows (Dry, high humidity): {schedules['pesticide_dates'][:3]}")

=== Run Summary for Maize ===
Predicted Flowering Date: 2023-07-01
Predicted Maturity Date: 2023-08-12
Recommended Fertilizer Windows: ['2023-05-30', '2023-06-01', '2023-06-04']
Recommended Pesticide Windows (Dry, high humidity): []


### 5.1 Extracting Irrigation (Watering) and Harvest (Yield) Schedules

We can query our simulation results to find:
1. **Irrigation Days:** When the soil water drops below a critical threshold (e.g., less than 35% of the soil's maximum capacity).
2. **Fully Grown Harvest Date:** The day the crop shifts to the 'Maturity' stage based on thermal heat unit accumulation.

In [9]:
def get_irrigation_and_harvest(sim_df, max_capacity):
    # Trigger irrigation if soil moisture drops below 40% of max capacity
    trigger_threshold = max_capacity * 0.40

    irrigation_needed = sim_df[sim_df['Soil_Water_mm'] < trigger_threshold]
    irrigation_dates = irrigation_needed['Date'].dt.strftime('%Y-%m-%d').tolist()

    # Harvest is the very first day the crop hits physiological maturity
    harvest_row = sim_df[sim_df['Stage'] == 'Maturity'].head(1)
    harvest_date = harvest_row['Date'].dt.strftime('%Y-%m-%d').values[0] if not harvest_row.empty else "Not reached"

    return {
        'irrigation_dates': irrigation_dates,
        'harvest_date': harvest_date
    }

# Run extraction on our simulation
water_and_harvest = get_irrigation_and_harvest(sim_results_df, soil_parameters['max_water_capacity'])

print(f"🌾 Crop: {crop_selected}")
print(f"📅 Estimated Harvest Date (Fully Grown): {water_and_harvest['harvest_date']}")
print(f"💧 Suggested Irrigation (Watering) Dates (First 5): {water_and_harvest['irrigation_dates'][:5]}")

🌾 Crop: Maize
📅 Estimated Harvest Date (Fully Grown): 2023-08-12
💧 Suggested Irrigation (Watering) Dates (First 5): ['2023-06-02', '2023-06-03', '2023-06-04', '2023-06-05', '2023-06-06']


### 7. Building a Crop Calendar Web API with FastAPI

To allow other systems or mobile apps to query this model, we can build an API endpoint. Let's first install `fastapi`, `uvicorn`, and `nest-asyncio` (required to run web servers inside Colab notebooks).

In [ ]:
!pip install -q fastapi uvicorn nest-asyncio

### 6. Crop Calendar Web API with FastAPI

We will update the FastAPI application endpoint to accept parameters for coordinates, crop type, and soil characteristics, run our offline crop model, and return a tailored crop calendar.

In [5]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import requests
import pandas as pd
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()

app = FastAPI(title="Offline-Ready Crop Calendar API")

class CropCalendarRequest(BaseModel):
    latitude: float
    longitude: float
    start_date: str
    end_date: str
    crop_type: str # e.g., 'Maize' or 'Wheat'
    max_water_capacity: float = 150.0
    initial_water_content: float = 75.0

@app.post("/predict_calendar")
def predict_calendar(req: CropCalendarRequest):
    if req.crop_type not in crop_parameters:
        raise HTTPException(status_code=400, detail=f"Crop type '{req.crop_type}' is not supported.")

    # 1. Fetch historical weather data
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={req.latitude}&longitude={req.longitude}&start_date={req.start_date}&end_date={req.end_date}&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_mean&timezone=auto"
    response = requests.get(url)
    if response.status_code != 200:
        raise HTTPException(status_code=400, detail="Failed to fetch weather data.")

    daily_data = response.json().get('daily', {})
    df = pd.DataFrame({
        'Date': pd.to_datetime(daily_data['time']),
        'Tmax': daily_data['temperature_2m_max'],
        'Tmin': daily_data['temperature_2m_min'],
        'Rainfall': daily_data['precipitation_sum'],
        'Humidity': daily_data['relative_humidity_2m_mean']
    })

    # 2. Run simulation offline with local crop/soil logic
    soil_settings = {
        'max_water_capacity': req.max_water_capacity,
        'initial_water_content': req.initial_water_content
    }

    sim_df, flowering_dt, maturity_dt = run_crop_simulation(df, req.crop_type, soil_settings)
    scheds = get_management_schedule(sim_df)

    return {
        "crop": req.crop_type,
        "soil_initial_water_mm": req.initial_water_content,
        "flowering_date": flowering_dt.strftime('%Y-%m-%d') if flowering_dt else None,
        "maturity_date": maturity_dt.strftime('%Y-%m-%d') if maturity_dt else None,
        "management_recommendations": {
            "fertilizer_application_dates": scheds['fertilizer_dates'],
            "optimal_pesticide_spray_dates": scheds['pesticide_dates']
        }
    }

# Restart server background thread
def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000)

import sys
# Small helper to restart background threads if they already exist
try:
    server_thread.join(timeout=0.1)
except NameError:
    pass

server_thread = Thread(target=run_api, daemon=True)
server_thread.start()
print("FastAPI app re-started on http://127.0.0.1:8000")

FastAPI app re-started on http://127.0.0.1:8000


### 9. Test the API

We can test our active local API directly by executing a `POST` request to local endpoint `http://127.0.0.1:8000/predict_calendar`.

In [8]:
import time
import json
time.sleep(2)

payload = {
    "latitude": 42.03,
    "longitude": -93.62,
    "start_date": "2023-05-01",
    "end_date": "2023-08-31",
    "crop_type": "Maize",
    "max_water_capacity": 150.0,
    "initial_water_content": 80.0
}

api_response = requests.post("http://127.0.0.1:8000/predict_calendar", json=payload)
print("API Response Body:")
print(json.dumps(api_response.json(), indent=2))

INFO:     127.0.0.1:56656 - "POST /predict_calendar HTTP/1.1" 200 OK
API Response Body:
{
  "crop": "Maize",
  "soil_initial_water_mm": 80.0,
  "flowering_date": "2023-07-01",
  "maturity_date": "2023-08-12",
  "management_recommendations": {
    "fertilizer_application_dates": [
      "2023-05-30",
      "2023-06-01",
      "2023-06-04",
      "2023-06-07",
      "2023-06-10",
      "2023-06-11"
    ],
    "optimal_pesticide_spray_dates": []
  }
}
